# 06 — Predicción Final

**Objetivo del notebook:** generar las predicciones sobre el conjunto de datos **sin etiquetar** (`smoking_prediction_entrega.xlsx`).

Este es el entregable final del proyecto. El paso crítico es aplicar **exactamente el mismo pipeline** de preparación que usamos en el entrenamiento — el mismo encoder, el mismo orden de columnas, el mismo modelo y el mismo threshold. Por eso cargamos todos los objetos guardados en lugar de re-crearlos.

Pasos:
1. Cargar los datos sin etiquetar.
2. Cargar los transformadores, el modelo y el threshold guardados.
3. Aplicar el pipeline idéntico al del entrenamiento.
4. Generar predicciones y exportar con la columna `smoking_prediction`.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas ✓')

Librerías cargadas ✓


## 1. Carga de los datos sin etiquetar

In [2]:
df_entrega = pd.read_excel('../data/raw/smoking_prediction_entrega.xlsx')
print(f'Datos de entrega: {df_entrega.shape[0]:,} filas × {df_entrega.shape[1]} columnas')
df_entrega.head()

Datos de entrega: 5,692 filas × 26 columnas


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,3.708333,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,6.625000,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y


## 2. Carga del pipeline entrenado

Cargamos todos los objetos que guardamos durante el entrenamiento. Esto garantiza que los datos nuevos se procesen de forma idéntica a los de entrenamiento — usar transformadores re-ajustados sobre los datos nuevos sería un error grave que invalidaría las predicciones.

In [3]:
ohe = joblib.load('../models/ohe_encoder.joblib')
metadata = joblib.load('../models/metadata_preprocesamiento.joblib')
modelo = joblib.load('../models/modelo_final_xgboost.joblib')
threshold = joblib.load('../models/threshold_final.joblib')

cat_cols = metadata['cat_cols']
num_cols = metadata['num_cols']
columnas_eliminadas = metadata['columnas_eliminadas']
columnas_finales = metadata['columnas_finales']

print('Pipeline cargado:')
print(f'  Encoder: OneHotEncoder ajustado')
print(f'  Modelo: XGBoost optimizado')
print(f'  Threshold: {threshold:.2f}')
print(f'  Columnas a eliminar: {columnas_eliminadas}')

Pipeline cargado:
  Encoder: OneHotEncoder ajustado
  Modelo: XGBoost optimizado
  Threshold: 0.42
  Columnas a eliminar: ['ID', 'oral']


## 3. Aplicación del pipeline

Replicamos paso a paso el mismo preprocesamiento del notebook 03, pero usando los transformadores ya ajustados (solo `transform`, nunca `fit`).

In [4]:
# Guardamos los IDs antes de eliminarlos, para el archivo final
ids = df_entrega['ID'].copy()

# Eliminamos las mismas columnas que en el entrenamiento
df_proc = df_entrega.drop(columns=columnas_eliminadas)

# Codificamos las categóricas con el encoder YA ajustado (solo transform)
ohe_arr = ohe.transform(df_proc[cat_cols])
ohe_df = pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(cat_cols), index=df_proc.index)

# Combinamos numéricas + categóricas codificadas
X_entrega = pd.concat([df_proc[num_cols], ohe_df], axis=1)

# Reordenamos las columnas EXACTAMENTE como en el entrenamiento
X_entrega = X_entrega[columnas_finales]

print(f'Datos procesados: {X_entrega.shape}')
print(f'¿Columnas coinciden con el entrenamiento?: {list(X_entrega.columns) == columnas_finales}')

Datos procesados: (5692, 26)
¿Columnas coinciden con el entrenamiento?: True


El reordenamiento de columnas con `X_entrega[columnas_finales]` es crítico: XGBoost espera las features en el mismo orden en que fue entrenado. Si el orden difiriera, las predicciones serían incorrectas aunque no diera error.

## 4. Generación de predicciones

Aplicamos el modelo y el threshold elegido en la validación. Usamos `predict_proba` y comparamos contra el threshold en lugar de `predict` directo, porque `predict` siempre usa 0.5 — y nosotros elegimos un threshold distinto.

In [5]:
# Probabilidad de la clase 1 (fuma)
proba = modelo.predict_proba(X_entrega)[:, 1]

# Aplicamos el threshold elegido en la validación
predicciones = (proba >= threshold).astype(int)

print(f'Predicciones generadas: {len(predicciones):,}')
print()
print('Distribución de predicciones:')
dist = pd.Series(predicciones).value_counts().sort_index()
print(f'  No fuma (0): {dist[0]:,} ({dist[0]/len(predicciones)*100:.1f}%)')
print(f'  Fuma (1):    {dist[1]:,} ({dist[1]/len(predicciones)*100:.1f}%)')

Predicciones generadas: 5,692

Distribución de predicciones:
  No fuma (0): 3,014 (53.0%)
  Fuma (1):    2,678 (47.0%)


## 5. Construcción y exportación del archivo final

Generamos el archivo de entrega: el dataset original más la nueva columna `smoking_prediction` con los valores como `0` o `1`, según pide la consigna.

In [6]:
# Partimos del dataset original e incorporamos la columna de predicción
df_resultado = df_entrega.copy()
df_resultado['smoking_prediction'] = predicciones

print('Vista del resultado:')
df_resultado[['ID', 'gender', 'age', 'smoking_prediction']].head(10)

Vista del resultado:


,ID,gender,age,smoking_prediction
0,27358,M,25,0
1,27364,M,30,1
2,27368,M,55,1
3,27378,M,20,0
4,27381,M,25,1
5,27386,M,45,1
6,27388,M,55,0
7,27393,F,50,0
8,27406,F,45,0
9,27417,M,40,1


In [7]:
# Exportamos a la carpeta processed
ruta_salida = '../data/processed/smoking_prediction_resultado.xlsx'
df_resultado.to_excel(ruta_salida, index=False)

print(f'Archivo exportado: {ruta_salida}')
print(f'Filas: {len(df_resultado):,}')
print(f'Columna de predicción: smoking_prediction (valores {sorted(df_resultado["smoking_prediction"].unique())})')

Archivo exportado: ../data/processed/smoking_prediction_resultado.xlsx
Filas: 5,692
Columna de predicción: smoking_prediction (valores [np.int64(0), np.int64(1)])


## Conclusiones de la predicción

- Cargamos los datos sin etiquetar (5.692 filas) y les aplicamos el **mismo pipeline exacto** del entrenamiento.
- Usamos los transformadores ajustados (`transform`, nunca `fit`) para garantizar consistencia.
- Reordenamos las columnas igual que en el entrenamiento, paso necesario para que el modelo prediga correctamente.
- Aplicamos el modelo XGBoost optimizado con el threshold elegido en la validación.
- Exportamos el archivo final con la columna `smoking_prediction` en formato `0`/`1`.

El archivo `smoking_prediction_resultado.xlsx` es el entregable que se evalúa con F1-Score sobre la clase 1.